In [1]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from statsmodels.tsa.seasonal import STL
import warnings
warnings.filterwarnings('ignore')

BASE = r"C:\Users\izouk\OneDrive\Desktop\projet aeroport\airline_performance"
BASE_SLASH = BASE.replace("\\", "/")

con = duckdb.connect()

df_monthly      = con.execute(f"SELECT * FROM read_parquet('{BASE_SLASH}/data/gold/mart_otp_monthly.parquet')").df()
df_seasonality  = con.execute(f"SELECT * FROM read_parquet('{BASE_SLASH}/data/gold/mart_seasonality.parquet')").df()

print("mart_otp_monthly  :", df_monthly.shape)
print("mart_seasonality  :", df_seasonality.shape)
print(df_monthly.head())

mart_otp_monthly  : (1798, 12)
mart_seasonality  : (2017, 10)
  carrier_key  YEAR  MONTH  season  total_flights  otp_pct  avg_arr_delay  \
0          9E  2009      1  Winter          22287    80.78          28.84   
1          9E  2009      2  Winter          20200    88.75          26.75   
2          9E  2009      3  Spring          22575    87.48          29.41   
3          9E  2009      4  Spring          22039    87.70          24.71   
4          9E  2009      5  Spring          22023    87.83          22.83   

   cancellation_rate  min_carrier  min_weather  min_nas  min_late_aircraft  
0               3.13      64529.0      10256.0  71608.0            64318.0  
1               1.81      35471.0       8072.0  37813.0            33799.0  
2               2.37      54068.0       7294.0  40041.0            45947.0  
3               1.36      47870.0       7973.0  41392.0            37102.0  
4               0.84      42292.0       6679.0  41765.0            34315.0  


In [2]:
pivot = df_monthly.groupby(['YEAR','MONTH'])['otp_pct'].mean().reset_index()
pivot = pivot.pivot(index='YEAR', columns='MONTH', values='otp_pct')

pivot.columns = ['Jan','Fév','Mar','Avr','Mai','Jun',
                 'Jul','Aoû','Sep','Oct','Nov','Déc']

fig = px.imshow(
    pivot,
    color_continuous_scale='RdYlGn',
    aspect='auto',
    title='OTP moyen par année et mois (2009-2018)',
    labels=dict(x='Mois', y='Année', color='OTP %')
)

fig.update_layout(height=450)
fig.show()

In [3]:
monthly_agg = df_monthly.groupby(['YEAR','MONTH'])['otp_pct'].mean().reset_index()
monthly_agg['date'] = pd.to_datetime(
    monthly_agg[['YEAR','MONTH']].assign(DAY=1)
    .rename(columns={'YEAR':'year','MONTH':'month','DAY':'day'})
)
ts = monthly_agg.set_index('date').sort_index()['otp_pct']

stl    = STL(ts, period=12, robust=True)
result = stl.fit()

df_stl = pd.DataFrame({
    'date'       : ts.index,
    'observed'   : result.observed,
    'trend'      : result.trend,
    'seasonal'   : result.seasonal,
    'residual'   : result.resid
})

fig = go.Figure()
fig.add_scatter(x=df_stl['date'], y=df_stl['observed'], name='OTP observé',  line=dict(color='gray', width=1))
fig.add_scatter(x=df_stl['date'], y=df_stl['trend'],    name='Tendance',     line=dict(color='blue', width=2))
fig.add_scatter(x=df_stl['date'], y=df_stl['seasonal'], name='Saisonnalité', line=dict(color='orange', width=1.5))
fig.add_scatter(x=df_stl['date'], y=df_stl['residual'], name='Résidu',       line=dict(color='red', width=1))

fig.update_layout(
    title='Décomposition STL — OTP mensuel 2009-2018',
    height=500,
    xaxis_title='Date',
    yaxis_title='OTP %'
)
fig.show()

In [4]:
hourly = df_seasonality.groupby('dep_hour')['otp_pct'].mean().reset_index()
hourly = hourly.dropna().sort_values('dep_hour')

fig = px.bar(
    hourly,
    x='dep_hour',
    y='otp_pct',
    title='OTP moyen par heure de départ — les vols du matin sont plus ponctuels',
    labels={'dep_hour': 'Heure de départ', 'otp_pct': 'OTP %'},
    color='otp_pct',
    color_continuous_scale='RdYlGn'
)

fig.update_layout(height=400)
fig.show()

In [5]:
dow_map = {1:'Lundi', 2:'Mardi', 3:'Mercredi', 4:'Jeudi',
           5:'Vendredi', 6:'Samedi', 7:'Dimanche'}

daily = df_seasonality.groupby('DAY_OF_WEEK')['otp_pct'].mean().reset_index()
daily['jour'] = daily['DAY_OF_WEEK'].map(dow_map)

fig = px.bar(
    daily,
    x='jour',
    y='otp_pct',
    title='OTP moyen par jour de la semaine (2009-2018)',
    labels={'otp_pct': 'OTP %', 'jour': 'Jour'},
    color='otp_pct',
    color_continuous_scale='RdYlGn'
)

fig.update_layout(height=400)
fig.show()

In [6]:
monthly_causes = df_seasonality.groupby('MONTH')[['pct_weather','pct_nas','pct_carrier']].mean().reset_index()

mois = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']
monthly_causes['mois_label'] = monthly_causes['MONTH'].apply(lambda x: mois[x-1])

fig = go.Figure()
fig.add_bar(x=monthly_causes['mois_label'], y=monthly_causes['pct_weather'],
            name='Weather (pic hiver)', marker_color='#2196F3')
fig.add_bar(x=monthly_causes['mois_label'], y=monthly_causes['pct_nas'],
            name='NAS (pic été)',       marker_color='#FF9800')
fig.add_bar(x=monthly_causes['mois_label'], y=monthly_causes['pct_carrier'],
            name='Carrier',             marker_color='#F44336')

fig.update_layout(
    barmode='group',
    title='Saisonnalité des causes — Weather vs NAS vs Carrier',
    xaxis_title='Mois',
    yaxis_title='% du total retard',
    height=450
)
fig.show()

In [7]:
trend_annuel = df_monthly.groupby('YEAR')['otp_pct'].mean().reset_index()

import numpy as np
z    = np.polyfit(trend_annuel['YEAR'], trend_annuel['otp_pct'], 1)
p    = np.poly1d(z)
trend_annuel['regression'] = p(trend_annuel['YEAR'])

fig = go.Figure()
fig.add_scatter(x=trend_annuel['YEAR'], y=trend_annuel['otp_pct'],
                mode='lines+markers', name='OTP moyen', line=dict(color='blue', width=2))
fig.add_scatter(x=trend_annuel['YEAR'], y=trend_annuel['regression'],
                mode='lines', name='Tendance linéaire',
                line=dict(color='red', dash='dash', width=1.5))

contexte = {
    2009: 'Récession',
    2011: 'Fusions',
    2014: 'Bas kérosène',
    2017: 'Saturation hubs'
}

for annee, label in contexte.items():
    fig.add_annotation(
        x=annee,
        y=trend_annuel[trend_annuel['YEAR']==annee]['otp_pct'].values[0],
        text=label,
        showarrow=True,
        arrowhead=2,
        ax=0, ay=-35,
        font=dict(size=10, color='gray')
    )

fig.update_layout(
    title='Tendance long terme OTP sectoriel 2009-2018',
    xaxis_title='Année',
    yaxis_title='OTP %',
    height=450
)
fig.show()